In [11]:
import importlib
import DataMapArchitect as dma
from DataMapArchitect import DataMapArchitect
import custom_image_selecter as cis

importlib.reload(dma)
importlib.reload(cis)


<module 'custom_image_selecter' from 'c:\\Users\\ThinkPad\\Documents\\UNIVERSITY\\PFE2\\custom_image_selecter.py'>

In [ ]:
 # CONFIGURATION
DATA_PATH = "./data/CASIA_FASD_V3/DATA"
OUTPUT_JSON = [
    "./CASIA_FASD_V3_30percent_BMSelection.json",
    "./CASIA_FASD_V3_50percent_BMSelection.json",
    "./CASIA_FASD_V3_70percent_BMSelection.json",
    "./CASIA_FASD_V3_90percent_BMSelection.json",
]

# datamap creation

In [13]:
# Initialize Architect
DATA_PATH = "data/CASIA_FASD_V3/DATA"
OUTPUT_JSON = ["./data_maps/CASIA_FASD_V3_all.json"]
architect = DataMapArchitect()
r = 0.3
for output in OUTPUT_JSON:
    results = architect.create_map_parallel(
        dataroute=DATA_PATH,
        max_workers=8,
        keep_ratio=1.0, 
        filter_fn=dma.uniform_sampling,
        image_selection_config=None
    )
    architect.save_to_json(output, results)
    r += 0.2

Mapping Progress: 100%|██████████| 600/600 [00:03<00:00, 158.22it/s]



[SUCCESS] Map saved to: ./data_maps/CASIA_FASD_V3_all.json


# data pipline

In [ ]:
import DataLoaderPipeline as dplmodule
from DataLoaderPipeline import DataLoaderPipeline
import importlib
importlib.reload(dplmodule)

In [ ]:
DATA_PATH = "data/CASIA_FASD_V3/DATA"
DATA_MAP_PATH = "./CASIA_FASD_V3_224-224_10percent_BCMSelection.json"
all_subjects = [f"{i:02d}" for i in range(1, 51)]  # Generate subject IDs from '01' to '50'.
train_subs = all_subjects[:20]
val_subs = all_subjects[20:25] 
test_subs = all_subjects[20:] 

dlp = DataLoaderPipeline(data_map_path=DATA_MAP_PATH, data_path=DATA_PATH, pix_range=(-1.0,1.0), img_size=(224,224), batch_size=32)


train_ds = dlp.build_pipeline(train_subs, balanced=True, augment=True)
val_ds   = dlp.build_pipeline(val_subs, balanced=False, augment=False)
test_ds  = dlp.build_pipeline(test_subs, balanced=False, augment=False)
for ds in [train_ds, val_ds, test_ds]:
    dlp.audit_dataset(ds, batchs=20)